In [1]:
import pandas as pd
import numpy as np

# load dataset 

normal_dataset_path = '../dataset/clean/normal/normal.csv'

attack_0rtt_dataset_path = '../dataset/clean/tls/attack_0rtt_dataset.csv'
attack_heartbleed_dataset_path = '../dataset/clean/tls/attack_heartbleed_dataset.csv'


attack_cert_probe_dataset_path = '../dataset/clean/probe/attack_cert_probe_dataset.csv'
attack_crypto_probe_dataset_path = '../dataset/clean/probe/attack_crypto_probe_dataset.csv'
attack_cve_probe_dataset_path = '../dataset/clean/probe/attack_cve_probe_dataset.csv'
attack_protocol_probe_dataset_path = '../dataset/clean/probe/attack_protocol_probe_dataset.csv'

attack_goldeneye_dataset_path = '../dataset/clean/dos/attack_goldeneye_dataset.csv'
attack_hulk_dataset_path = '../dataset/clean/dos/attack_hulk_dataset.csv'
attack_rst_flood_dataset_path = '../dataset/clean/dos/attack_rst_flood_dataset.csv'
attack_slowloris_dataset_path = '../dataset/clean/dos/attack_slowloris_dataset.csv'
attack_sync_flood_dataset_path = '../dataset/clean/dos/attack_sync_flood_dataset.csv'
attack_tcp_ack_dataset_path = '../dataset/clean/dos/attack_tcp_ack_dataset.csv'
attack_tors_dataset_path = '../dataset/clean/dos/attack_tors_dataset.csv'
attack_udp_dataset_path = '../dataset/clean/dos/attack_udp_dataset.csv'




normal_dataset = pd.read_csv(normal_dataset_path)

attack_0rtt_dataset = pd.read_csv(attack_0rtt_dataset_path)
attack_cert_probe_dataset = pd.read_csv(attack_cert_probe_dataset_path)
attack_crypto_probe_dataset = pd.read_csv(attack_crypto_probe_dataset_path)
attack_cve_probe_dataset = pd.read_csv(attack_cve_probe_dataset_path)
attack_goldeneye_dataset = pd.read_csv(attack_goldeneye_dataset_path)
attack_heartbleed_dataset = pd.read_csv(attack_heartbleed_dataset_path)
attack_hulk_dataset = pd.read_csv(attack_hulk_dataset_path)
attack_protocol_probe_dataset = pd.read_csv(attack_protocol_probe_dataset_path)
attack_rst_flood_dataset = pd.read_csv(attack_rst_flood_dataset_path)
attack_slowloris_dataset = pd.read_csv(attack_slowloris_dataset_path)
attack_sync_flood_dataset = pd.read_csv(attack_sync_flood_dataset_path)
attack_tcp_ack_dataset = pd.read_csv(attack_tcp_ack_dataset_path)
attack_tors_dataset = pd.read_csv(attack_tors_dataset_path)
attack_udp_dataset = pd.read_csv(attack_udp_dataset_path)


# Randomly sample data

# Normal
normal_dataset = normal_dataset.sample(n=49000, random_state=42) 

# TLS (use full)
tls_dataset = pd.concat(
    [
        attack_0rtt_dataset,
        attack_heartbleed_dataset
    ],
    axis=0,
    ignore_index=True
)

# Probe 
probe_dataset = pd.concat(
    [
        attack_cert_probe_dataset.sample(n=1000, random_state=42),
        attack_crypto_probe_dataset.sample(n=1000, random_state=42) ,
        attack_cve_probe_dataset.sample(n=2000, random_state=42) ,
        attack_protocol_probe_dataset.sample(n=1000, random_state=42) 
    ],
    axis=0,
    ignore_index=True
)

# Dos
dos_dataset = pd.concat(
    [
        attack_goldeneye_dataset.sample(n=1000, random_state=42),
        attack_hulk_dataset.sample(n=1000, random_state=42),
        attack_rst_flood_dataset,
        attack_slowloris_dataset,
        attack_sync_flood_dataset.sample(n=1000, random_state=42),
        attack_tcp_ack_dataset.sample(n=1000, random_state=42),
        attack_tors_dataset.sample(n=1000, random_state=42),
        attack_udp_dataset
    ],
    axis=0,
    ignore_index=True
)



In [2]:
normal_df = normal_dataset
attack_df = pd.concat([tls_dataset, probe_dataset, dos_dataset], ignore_index=True)

# Combine all
dataset_df = pd.concat([normal_df, attack_df], ignore_index=True)

#  Shuffle the data
dataset_df = dataset_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Result
print("Combined shape:", dataset_df.shape)
print(dataset_df['label'].value_counts())

Combined shape: (59380, 77)
label
normal    49000
ddos       5264
probe      5000
tls         116
Name: count, dtype: int64


# IDS GRU

In [3]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, balanced_accuracy_score, f1_score
)

# ===== 1) Data =====
X = dataset_df.drop(columns=['label']).values.astype(np.float32)
y_str = dataset_df['label'].values

le = LabelEncoder()
y = le.fit_transform(y_str).astype(np.int64)
classes = le.classes_
num_classes = len(classes)
print("Classes:", classes)

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)

# 70/10/20 split
X_tr_full, X_te, y_tr_full, y_te = train_test_split(
    X_scaled, y, test_size=0.20, random_state=42, stratify=y
)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_tr_full, y_tr_full, test_size=0.125, random_state=42, stratify=y_tr_full
)

# Reshape to (B, T, F=1)
def to_seq(a): return torch.tensor(a, dtype=torch.float32).unsqueeze(-1)

X_tr_t, X_val_t, X_te_t = to_seq(X_tr), to_seq(X_val), to_seq(X_te)
y_tr_t, y_val_t, y_te_t = torch.tensor(y_tr), torch.tensor(y_val), torch.tensor(y_te)

train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=64, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=64, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_te_t, y_te_t),  batch_size=64, shuffle=False)

# ===== 2) Class weights =====
unique, counts = np.unique(y_tr, return_counts=True)
inv = {k: 1.0/v for k, v in dict(zip(unique, counts)).items()}
scale = np.mean(list(inv.values()))
class_weights = torch.tensor([inv[k]/scale for k in sorted(inv.keys())], dtype=torch.float32)

# ===== 3) GRU model (no extra head) =====
class GRUClassifier(nn.Module):
    def __init__(self, input_size=1, hidden_size=128, num_layers=2,
                 bidirectional=False, dropout=0.3, num_classes=4):
        super().__init__()
        self.bidirectional = bidirectional
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=(dropout if num_layers > 1 else 0.0),
            bidirectional=bidirectional
        )
        out_dim = hidden_size * (2 if bidirectional else 1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(out_dim, num_classes)

    def forward(self, x):
        _, h_n = self.gru(x)
        if self.bidirectional:
            h_last = torch.cat([h_n[-2], h_n[-1]], dim=1)
        else:
            h_last = h_n[-1]
        h_last = self.dropout(h_last)
        return self.fc(h_last)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GRUClassifier(
    input_size=1,
    hidden_size=128,
    num_layers=2,
    bidirectional=False,
    dropout=0.3,
    num_classes=num_classes
).to(device)

# ===== 4) Loss, optimizer =====
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
max_grad_norm = 2.0
epochs = 60
best_val_bal_acc = 0.0

# ===== 5) Training =====
for epoch in range(1, epochs + 1):
    model.train()
    tr_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()
        tr_loss += loss.item()

    # Validation
    model.eval()
    val_loss = 0.0
    val_true, val_pred = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            val_loss += loss.item()
            preds = logits.argmax(dim=1)
            val_true.extend(yb.cpu().numpy())
            val_pred.extend(preds.cpu().numpy())

    val_acc = accuracy_score(val_true, val_pred)
    val_bal_acc = balanced_accuracy_score(val_true, val_pred)

    if val_bal_acc > best_val_bal_acc:
        best_val_bal_acc = val_bal_acc
        torch.save(model.state_dict(), "best_gru.pth")

    print(f"Epoch [{epoch:03d}/{epochs}] "
          f"TrainLoss: {tr_loss/len(train_loader):.4f} "
          f"ValLoss: {val_loss/len(val_loader):.4f} "
          f"ValAcc: {val_acc:.4f} ValBalanced: {val_bal_acc:.4f}")

# ===== 6) Testing =====
model.load_state_dict(torch.load("best_gru.pth", map_location=device))
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        logits = model(xb)
        y_pred.extend(logits.argmax(dim=1).cpu().numpy())
        y_true.extend(yb.numpy())

print("TEST accuracy:", accuracy_score(y_true, y_pred))
print("TEST balanced_accuracy:", balanced_accuracy_score(y_true, y_pred))
print("TEST macro F1:", f1_score(y_true, y_pred, average="macro"))
print("\nConfusion Matrix:\n", confusion_matrix(y_true, y_pred))
print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=classes))

Classes: ['ddos' 'normal' 'probe' 'tls']
Epoch [001/60] TrainLoss: 0.6293 ValLoss: 0.3318 ValAcc: 0.9791 ValBalanced: 0.7780
Epoch [002/60] TrainLoss: 0.3308 ValLoss: 0.2415 ValAcc: 0.9801 ValBalanced: 0.7796
Epoch [003/60] TrainLoss: 0.3008 ValLoss: 0.2095 ValAcc: 0.9916 ValBalanced: 0.7826
Epoch [004/60] TrainLoss: 0.2648 ValLoss: 0.2189 ValAcc: 0.9911 ValBalanced: 0.8249
Epoch [005/60] TrainLoss: 0.2665 ValLoss: 0.2440 ValAcc: 0.9929 ValBalanced: 0.7843
Epoch [006/60] TrainLoss: 0.2276 ValLoss: 0.2204 ValAcc: 0.9936 ValBalanced: 0.7845
Epoch [007/60] TrainLoss: 0.2284 ValLoss: 0.3403 ValAcc: 0.9946 ValBalanced: 0.8061
Epoch [008/60] TrainLoss: 0.1918 ValLoss: 0.2192 ValAcc: 0.9924 ValBalanced: 0.7825
Epoch [009/60] TrainLoss: 0.1891 ValLoss: 0.1900 ValAcc: 0.9692 ValBalanced: 0.9023
Epoch [010/60] TrainLoss: 0.1757 ValLoss: 0.1853 ValAcc: 0.9808 ValBalanced: 0.9067
Epoch [011/60] TrainLoss: 0.1865 ValLoss: 0.1979 ValAcc: 0.9912 ValBalanced: 0.8041
Epoch [012/60] TrainLoss: 0.1524 Va

/tmp/ipykernel_343007/1413724798.py:136: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_gru.pth", map_location=device))
